In [1]:

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import seaborn as sns
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.regularizers import l2
import joblib
import keras_tuner as kt

# Cargar datos
data = pd.read_csv("C:/UABC/Topicos selectos de la investigacion/MatchesDataSet/merged_all_statistics.txt", sep=",")

# Crear etiquetas (1 si gana el equipo local, 0 en caso contrario)
y = (data['golesLocal'] > data['golesVisitante']).astype(int)

# Seleccionar características iniciales, eliminando identificadores y columnas irrelevantes
features_to_drop = ['idPartido', 'EquipoLocal', 'EquipoVisitante', 'golesLocal', 'golesVisitante', 'Temporada']
X = data.drop(columns=features_to_drop)

# Manejo de valores faltantes
imputer = SimpleImputer(strategy='mean')
X = imputer.fit_transform(X)

# Escalado de características
scaler = StandardScaler()
X = scaler.fit_transform(X)

# División de datos en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Balanceo de clases usando SMOTE
sm = SMOTE(random_state=42)
X_train_balanced, y_train_balanced = sm.fit_resample(X_train, y_train)

# Guardar preprocesadores para usar en el testeo
joblib.dump(imputer, 'imputer.pkl')
joblib.dump(scaler, 'scaler.pkl')

# Función para construir el modelo
def build_model(hp):
    model = Sequential()
    model.add(Dense(
        units=hp.Int('units_1', min_value=128, max_value=512, step=64),
        activation='relu',
        input_dim=X_train_balanced.shape[1],
        kernel_regularizer=l2(hp.Float('l2_1', min_value=0.0001, max_value=0.01, sampling='LOG'))
    ))
    model.add(Dropout(hp.Float('dropout_1', min_value=0.2, max_value=0.5, step=0.1)))
    model.add(Dense(
        units=hp.Int('units_2', min_value=64, max_value=256, step=64),
        activation='relu',
        kernel_regularizer=l2(hp.Float('l2_2', min_value=0.0001, max_value=0.01, sampling='LOG'))
    ))
    model.add(Dropout(hp.Float('dropout_2', min_value=0.2, max_value=0.5, step=0.1)))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer=hp.Choice('optimizer', values=['adam', 'nadam', 'rmsprop']),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Optimizador de hiperparámetros
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=50,
    directory='my_dir',
    project_name='soccer_model_tuning_v4'
)

# Búsqueda de hiperparámetros
tuner.search(X_train_balanced, y_train_balanced, epochs=50, validation_split=0.3, verbose=1)
best_hps = tuner.get_best_hyperparameters()[0]

# Construcción del modelo con los mejores hiperparámetros
model = tuner.hypermodel.build(best_hps)

# Callback para detener el entrenamiento temprano
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Entrenamiento del modelo
history = model.fit(
    X_train_balanced, y_train_balanced,
    validation_split=0.3,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)

# Guardar el modelo entrenado
model.save('soccer_model.h5')

# Evaluación del modelo en el conjunto de prueba
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy en el conjunto de prueba: {accuracy:.2f}")

# Métricas
y_pred_proba = model.predict(X_test).flatten()
y_pred = (y_pred_proba > 0.5).astype(int)
print("Reporte de clasificación:")
print(classification_report(y_test, y_pred))


KeyboardInterrupt: 

In [49]:
import os
import pandas as pd
import joblib
from tensorflow.keras.models import load_model

# Rutas de los archivos guardados
imputer_path = "C:/UABC/Topicos selectos de la investigacion/MatchesDataSet/training_model/neuronal_networks/imputer.pkl"
scaler_path = "C:/UABC/Topicos selectos de la investigacion/MatchesDataSet/training_model/neuronal_networks/scaler.pkl"
model_path = "C:/UABC/Topicos selectos de la investigacion/MatchesDataSet/training_model/neuronal_networks/soccer_model.h5"

# Verificar que los archivos existen
if not os.path.exists(imputer_path):
    raise FileNotFoundError(f"El archivo '{imputer_path}' no existe.")
if not os.path.exists(scaler_path):
    raise FileNotFoundError(f"El archivo '{scaler_path}' no existe.")
if not os.path.exists(model_path):
    raise FileNotFoundError(f"El archivo '{model_path}' no existe.")

# Cargar el modelo y los preprocesadores
imputer = joblib.load(imputer_path)
scaler = joblib.load(scaler_path)
model = load_model(model_path)

# Ruta del archivo del partido específico
archivo_partido = "C:/UABC/Topicos selectos de la investigacion/MatchesDataSet/current_season_data/j1/mallorca_realmadrid.csv"
partido_data = pd.read_csv(archivo_partido)

# Eliminar columnas irrelevantes
features_to_drop = ['idPartido', 'EquipoLocal', 'EquipoVisitante', 'golesLocal', 'golesVisitante', 'Temporada']
partido_data = partido_data.drop(columns=features_to_drop, errors='ignore')

# Preprocesamiento del partido
# Manejo de valores faltantes (usando el imputador entrenado)
partido_data = imputer.transform(partido_data)

# Escalado (usando el escalador entrenado)
partido_data_scaled = scaler.transform(partido_data)

# Generar predicción
prediccion_probabilidad = model.predict(partido_data_scaled).flatten()
prediccion_binaria = (prediccion_probabilidad > 0.5).astype(int)

# Mostrar resultados
for i, prob in enumerate(prediccion_probabilidad):
    print(f"Partido {i+1}:")
    print(f" - Probabilidad de victoria del equipo local: {prob * 100:.2f}%")
    print(f" - Probabilidad del empate/victoria del equipo visitante: {(1 - prob) * 100:.2f}%")
    print(f" - Predicción binaria: {'Gana' if prediccion_binaria[i] == 1 else 'No gana'}\n")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step
Partido 1:
 - Probabilidad de victoria del equipo local: 2.57%
 - Probabilidad del empate/victoria del equipo visitante: 97.43%
 - Predicción binaria: No gana

